<a href="https://colab.research.google.com/github/kab-s/flyrank-page-priority-pipeline/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)


## 1. Ranked actions + reason codes

### The Operational Challenge: Triage Under Finite Capacity
In an enterprise search portfolio comprising tens of thousands of articles across dozens of publishing domains, traffic volatility is continuous. Ranking pages solely by predicted risk ($P(\text{decline})$) floods the triage queue with low-traffic tail pages that drop from 3 impressions to 1—an outcome with negligible business consequence. Conversely, sorting solely by historical traffic volume prioritizes stable pillar pages that are not at risk, wasting scarce editorial attention.

### Two-Part Expected Utility Framework
To allocate editorial capacity where real business value is concentrated, here is a  synthesis of **Two-Part Expected Utility Ranking Framework**:
$$\text{Priority Score} = P(\text{decline in next 30 days}) \times \ln(1 + \text{impr}_{w30}) \times (1 + \max(0, \Delta \text{pos}))$$

- **Risk ($P(\text{decline})$):** Estimated via `HistGradientBoostingClassifier`, calibrated out-of-fold across unseen publishers via 5-fold `GroupKFold` cross-validation with inverse client frequency weighting (ROC-AUC 0.6505, PR-AUC 0.2962, Brier score 0.1263).
- **Expected Impact:** Quantified as **Expected Clicks Lost** = $\text{Baseline Clicks} \times \text{empirical_drop_rate}$, where the drop severity is dynamically estimated from retrospective decaying pages preceding $T$ (zero future leakage; fully portable to any snapshot date), and baseline clicks blend historical traffic with an empirical-Bayes position prior curve ($m = 100$).
- **Operational Ranking Score:** Evaluated via $\hat{p} \times \ln(1 + \text{impr}_{w30}) \times (1 + \max(0, \Delta \text{pos}))$, balancing probability of loss, audience scale, and search ranking trajectory.

### Action Taxonomy: "Model = Order, Rule = Reason"
Every article in the queue is assigned a reason code and an actionable tactical SEO recommendation:
1. `Immediate Content Refresh & SERP Intent Update`: Triggered by severe velocity drop (<0.50x) and rank slip (>=0.5 ranks). Urgent content revision, factual refresh, and schema alignment.
2. `Snippet & Meta Tag CTR Optimization`: Triggered by severe velocity drop (<0.50x) while SERP rank held steady. Audit snippet truncation, search feature displacement, and click-through rates.
3. `Internal Linking & Authority Reinforcement`: Triggered by moderate decay (0.50x–0.80x) with rank slippage (>=0.5 ranks). Re-route internal PageRank and contextual anchor equity from high-authority category hubs.
4. `Comprehensive Section Expansion`: Triggered by moderate decay (0.50x–0.80x) with stable rank. Deepen secondary topic coverage, add FAQ modules, and address long-tail query variance.
5. `Floor Stabilization Review / Cooldown Watchlist`: Triggered by sub-floor decay (<500 impressions) or historical collapse near structural floor (Failure Archetype 1). Batch or monitor; do not consume senior copywriter hours.
6. `Quiet Erosion Deep-Dive`: Triggered when heuristic rules see no decay (ratio >= 0.80) but the multi-signal ML model detects elevated risk ($\hat{p} \ge 0.40$). Uncovers subtle multi-factor erosion before precipitous drop occurs.
7. `Passive Monitoring`: Triggered for healthy, stable, or rising articles. Protected from unneeded edits to preserve organic momentum.
8. `Tracking & Query Verification`: Triggered by extreme sudden surges (>7x or >20x). Audit analytics instrumentation and PR campaigns before editing.

In [1]:
# Code Block 1: Operational ML Priority Queue & Action Taxonomy Synthesis
# Evaluates 38,271 eligible page snapshots at T = 2026-02-01 across 22 client domains
import os
import warnings
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Dynamic Warehouse Source Configuration
# Seamless support for:
#   1. Google Colab (remote HuggingFace via Colab Secrets)
#   2. Local Workspace (pre-downloaded Parquet warehouse)
#   3. Generic Remote (HuggingFace via environment HF_TOKEN)
# ---------------------------------------------------------------------------
def get_warehouse_source():
    # 1. Check if running inside Google Colab
    try:
        from google.colab import userdata
        try:
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = os.environ.get('HF_TOKEN')
        return 'colab', 'hf://datasets/FlyRank/internship-warehouse', hf_token
    except ImportError:
        pass

    # 2. Check if local warehouse exists on disk
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        for candidate in [p / 'scripts' / 'warehouse', p / 'warehouse']:
            if (candidate / 'dim_clients.parquet').exists():
                return 'local', candidate.resolve().as_posix(), None

    # 3. Fallback: remote Hugging Face with environment variable
    hf_token = os.environ.get('HF_TOKEN')
    return 'remote_hf', 'hf://datasets/FlyRank/internship-warehouse', hf_token

env_mode, REL, HF_TOKEN = get_warehouse_source()
print(f"Warehouse Source Mode: {env_mode.upper()} -> {REL}")

wh_dir = REL  # Retain backward compatibility alias for existing code
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")

# Configure Hugging Face Secret if connecting to remote dataset
if REL.startswith('hf://') and HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Parquet paths (portable across local disk and Hugging Face remote)
dim_clients_path = f"{REL}/dim_clients.parquet"
dim_content_path = f"{REL}/dim_content.parquet"
fact_daily_path = f"{REL}/fact_content_daily_performance/**/*.parquet"

con.execute(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")
con.execute(f"CREATE OR REPLACE VIEW fact_daily AS SELECT * FROM read_parquet('{fact_daily_path}')")

# 2. Compute Historical CTR Curve by Position Bucket for Empirical Bayes Baseline
ctr_query = """
SELECT
    ROUND(LEAST(GREATEST(gsc_sum_position / NULLIF(gsc_impressions, 0), 1.0), 20.0))::INT AS pos_bucket,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
FROM fact_daily
WHERE report_date <= DATE '2026-02-01'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
GROUP BY 1 ORDER BY 1
"""
ctr_curve_df = con.sql(ctr_query).df()
ctr_dict = dict(zip(ctr_curve_df["pos_bucket"], ctr_curve_df["ctr"]))
default_ctr = float(ctr_curve_df["ctr"].mean())

# 3. Extract Eligible Snapshot Cohort at T = 2026-02-01
extraction_sql = """
WITH daily_scoped AS (
    SELECT
        content_hash_id, client_hash_id, report_date,
        gsc_data_available, ga4_data_available,
        gsc_impressions, gsc_clicks, gsc_sum_position,
        ga4_sessions, ga4_engaged_sessions
    FROM fact_daily
    WHERE report_date >= DATE '2026-02-01' - INTERVAL 270 DAY
      AND report_date <= DATE '2026-02-01' + INTERVAL 30 DAY
),
page_windows AS (
    SELECT
        content_hash_id, client_hash_id,
        COUNT(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' AND gsc_data_available IS TRUE THEN 1 END) AS gsc_days_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' THEN gsc_impressions ELSE 0 END) AS impr_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' THEN gsc_clicks ELSE 0 END) AS clicks_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' THEN gsc_sum_position ELSE 0 END) AS sum_pos_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 59 DAY AND report_date <= DATE '2026-02-01' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS impr_prev30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 59 DAY AND report_date <= DATE '2026-02-01' - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END) AS clicks_prev30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 59 DAY AND report_date <= DATE '2026-02-01' - INTERVAL 30 DAY THEN gsc_sum_position ELSE 0 END) AS sum_pos_prev30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 89 DAY AND report_date <= DATE '2026-02-01' THEN gsc_impressions ELSE 0 END) AS impr_w90,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 89 DAY AND report_date <= DATE '2026-02-01' THEN gsc_clicks ELSE 0 END) AS clicks_w90,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 89 DAY AND report_date <= DATE '2026-02-01' THEN gsc_sum_position ELSE 0 END) AS sum_pos_w90,
        COUNT(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 89 DAY AND report_date <= DATE '2026-02-01' AND gsc_impressions > 0 THEN 1 END) AS days_with_impr_w90,
        MAX(CASE WHEN report_date <= DATE '2026-02-01' THEN gsc_impressions ELSE 0 END) AS max_daily_impr_up_to_T,
        MAX(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_observed_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' THEN ga4_sessions ELSE 0 END) AS ga4_sessions_w30,
        SUM(CASE WHEN report_date >= DATE '2026-02-01' - INTERVAL 29 DAY AND report_date <= DATE '2026-02-01' THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_w30,
        COUNT(CASE WHEN report_date > DATE '2026-02-01' AND report_date <= DATE '2026-02-01' + INTERVAL 30 DAY AND gsc_data_available IS TRUE THEN 1 END) AS gsc_days_forward,
        SUM(CASE WHEN report_date > DATE '2026-02-01' AND report_date <= DATE '2026-02-01' + INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS impr_forward
    FROM daily_scoped
    GROUP BY content_hash_id, client_hash_id
),
client_totals AS (
    SELECT client_hash_id, SUM(impr_w30) AS client_impr_w30, SUM(impr_prev30) AS client_impr_prev30
    FROM page_windows GROUP BY client_hash_id
)
SELECT
    pw.content_hash_id, pw.client_hash_id,
    DATE '2026-02-01' AS snapshot_date,
    pw.impr_w30, pw.clicks_w30, pw.impr_prev30, pw.clicks_prev30, pw.impr_w90, pw.clicks_w90,
    COALESCE(pw.sum_pos_w90 / NULLIF(pw.impr_w90, 0), 0.0) AS position_w90, pw.impr_forward,
    CASE WHEN pw.impr_forward < 0.80 * pw.impr_w30 THEN 1 ELSE 0 END AS is_declining_label,
    LN(1 + pw.impr_w30) AS log_impr_w30, LN(1 + pw.clicks_w30) AS log_clicks_w30,
    COALESCE(pw.clicks_w30 / NULLIF(pw.impr_w30, 0), 0.0) AS ctr_w30,
    COALESCE(pw.sum_pos_w30 / NULLIF(pw.impr_w30, 0), 0.0) AS position_w30,
    pw.days_with_impr_w90, CASE WHEN pw.clicks_w90 > 0 THEN 1 ELSE 0 END AS has_clicks_w90,
    COALESCE(LEAST(GREATEST(pw.impr_w30 / NULLIF(pw.impr_prev30, 0), 0.0), 10.0), 10.0) AS impr_ratio_30_prev30,
    CASE WHEN pw.impr_prev30 = 0 THEN 1 ELSE 0 END AS prev30_no_impr_flag,
    COALESCE(LEAST(GREATEST(pw.clicks_w30 / NULLIF(pw.clicks_prev30, 0), 0.0), 10.0), 10.0) AS clicks_ratio_30_prev30,
    COALESCE((pw.sum_pos_w30 / NULLIF(pw.impr_w30, 0)) - (pw.sum_pos_prev30 / NULLIF(pw.impr_prev30, 0)), 0.0) AS position_change_30_prev30,
    COALESCE((pw.clicks_w30 / NULLIF(pw.impr_w30, 0)) - (pw.clicks_prev30 / NULLIF(pw.clicks_prev30, 0)), 0.0) AS ctr_change_30_prev30,
    COALESCE(LEAST(GREATEST(pw.impr_w30 / NULLIF(pw.max_daily_impr_up_to_T * 30.0, 0), 0.0), 1.0), 1.0) AS pct_of_peak_proxy,
    COALESCE(pw.impr_w30 / NULLIF(pw.impr_prev30, 0), 1.0) AS raw_impr_ratio,
    DATE_DIFF('day', dc.content_created_date, DATE '2026-02-01') AS age_days,
    COALESCE(dc.content_type, 'unspecified') AS content_type,
    COALESCE(dc.main_intent, 'unspecified') AS main_intent,
    LN(1 + COALESCE(dc.search_volume, 0)) AS log_search_volume,
    CASE WHEN dc.search_volume IS NULL THEN 1 ELSE 0 END AS search_volume_missing,
    COALESCE(dc.cpc, 0.0) AS cpc,
    CASE WHEN dc.cpc IS NULL THEN 1 ELSE 0 END AS cpc_missing,
    LN(1 + COALESCE(dc.backlinks, 0)) AS log_backlinks,
    CASE WHEN dc.backlinks IS NULL THEN 1 ELSE 0 END AS backlinks_missing,
    LN(1 + COALESCE(dc.word_count, 0)) AS log_word_count,
    CASE WHEN dc.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
    LN(1 + COALESCE(dc.char_count, 0)) AS log_char_count,
    pw.ga4_observed_w30,
    COALESCE(pw.ga4_engaged_w30 / NULLIF(pw.ga4_sessions_w30, 0), 0.0) AS engagement_rate_w30,
    LN(1 + COALESCE(pw.ga4_sessions_w30, 0)) AS log_ga4_sessions_w30,
    COALESCE(LEAST(GREATEST(ct.client_impr_w30 / NULLIF(ct.client_impr_prev30, 0), 0.0), 10.0), 1.0) AS client_impr_ratio_30_prev30,
    COALESCE(pw.impr_w30 / NULLIF(ct.client_impr_w30, 0), 0.0) AS client_share_of_traffic
FROM page_windows pw
JOIN dim_content dc ON pw.content_hash_id = dc.content_hash_id
JOIN dim_clients dcl ON pw.client_hash_id = dcl.client_hash_id
JOIN client_totals ct ON pw.client_hash_id = ct.client_hash_id
WHERE dc.is_published IS TRUE AND dc.is_deleted IS NOT TRUE
  AND DATE_DIFF('day', dc.content_created_date, DATE '2026-02-01') >= 90
  AND DATE_DIFF('day', dcl.gsc_data_start, DATE '2026-02-01') >= 90
  AND pw.gsc_days_w30 >= 25
  AND pw.gsc_days_forward >= 25
  AND pw.impr_w30 >= 100
"""
df = con.sql(extraction_sql).df()
# Enforce deterministic row order across all OS environments
df = df.sort_values(by=["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["snapshot_date"] = "2026-02-01"
n_rows = len(df)
n_clients = df["client_hash_id"].nunique()
base_rate = float(df["is_declining_label"].mean())
print(f"Extracted {n_rows:,} eligible page snapshots across {n_clients} clients (Base Rate: {base_rate*100:.2f}%)")

# 4. Out-of-Fold Trained Champion Model (HistGradientBoosting)
numeric_features = [
    "log_impr_w30", "log_clicks_w30", "ctr_w30", "position_w30", "days_with_impr_w90", "has_clicks_w90",
    "impr_ratio_30_prev30", "prev30_no_impr_flag", "clicks_ratio_30_prev30",
    "position_change_30_prev30", "ctr_change_30_prev30", "pct_of_peak_proxy",
    "age_days", "log_search_volume", "search_volume_missing", "cpc", "cpc_missing",
    "log_backlinks", "backlinks_missing", "log_word_count", "word_count_missing", "log_char_count",
    "ga4_observed_w30", "engagement_rate_w30", "log_ga4_sessions_w30",
    "client_impr_ratio_30_prev30", "client_share_of_traffic"
]
categorical_features = ["content_type", "main_intent"]
all_features = numeric_features + categorical_features

hgb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
    ],
    remainder="drop"
)
cat_indices = [len(numeric_features), len(numeric_features) + 1]

# Deterministic seeded GroupKFold client assignment (seed 42)
unique_clients = np.sort(df["client_hash_id"].unique())
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
client_to_fold = {client: i % 5 for i, client in enumerate(shuffled_clients)}
fold_assignment = df["client_hash_id"].map(client_to_fold).values

oof_prob_hgb = np.zeros(n_rows, dtype=np.float64)
y = df["is_declining_label"].values

for fold in range(5):
    train_idx = np.where(fold_assignment != fold)[0]
    test_idx = np.where(fold_assignment == fold)[0]
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    train_clients_series = df_train["client_hash_id"]
    client_sizes = train_clients_series.value_counts()
    client_weights = 1.0 / np.sqrt(train_clients_series.map(client_sizes).values)
    client_weights = client_weights / client_weights.mean()

    hgb_classifier = HistGradientBoostingClassifier(
        categorical_features=cat_indices,
        random_state=42, max_iter=250, min_samples_leaf=40, learning_rate=0.08, l2_regularization=1.0
    )

    X_train_trans = hgb_preprocessor.fit_transform(df_train[all_features])
    X_test_trans = hgb_preprocessor.transform(df_test[all_features])
    hgb_classifier.fit(X_train_trans, y[train_idx], sample_weight=client_weights)
    oof_prob_hgb[test_idx] = hgb_classifier.predict_proba(X_test_trans)[:, 1]

df["risk_prob_hgb"] = oof_prob_hgb

# 5. Compute Impact and Operational ML Priority Score
m = 100.0
pos_buckets = np.clip(np.round(np.where(df["position_w90"] > 0, df["position_w90"], df["position_w30"])), 1, 20).astype(int)
priors = np.array([ctr_dict.get(p, default_ctr) for p in pos_buckets])
df["ctr_smoothed"] = (df["clicks_w90"].values + m * priors) / (df["impr_w90"].values + m)
df["baseline_clicks"] = df["impr_w30"].values * df["ctr_smoothed"].values

# Dynamic Retrospective Empirical Drop Rate (Zero Leakage, Adapts to Any Snapshot T)
retro_decliners = df[(df["impr_w30"] < 0.80 * df["impr_prev30"]) & (df["impr_prev30"] > 0)]
if len(retro_decliners) >= 30:
    empirical_drop_rate = float(
        ((retro_decliners["impr_prev30"] - retro_decliners["impr_w30"]) / retro_decliners["impr_prev30"]).median()
    )
else:
    empirical_drop_rate = 0.20
df["expected_clicks_lost"] = df["baseline_clicks"].values * empirical_drop_rate
print(f"Dynamic Retrospective Drop Rate at T: {empirical_drop_rate*100:.2f}% (derived from {len(retro_decliners):,} decaying pages)")

df["ml_priority_score"] = (
    df["risk_prob_hgb"].values
    * df["log_impr_w30"].values
    * (1.0 + np.maximum(0.0, df["position_change_30_prev30"].values))
)

# Baseline Rule for Direct Comparison (ML-07)
baseline_active = (df["impr_w30"] >= 500) & (df["impr_ratio_30_prev30"] < 0.80)
df["baseline_action_score"] = np.where(
    baseline_active,
    (1.0 - np.minimum(0.80, df["impr_ratio_30_prev30"].values))
    * df["log_impr_w30"].values
    * (1.0 + np.maximum(0.0, df["position_change_30_prev30"].values)),
    0.0
)

# 6. Assign Authentic Phase 4 Reason Codes & Actions with Gate Precedence
reason_codes, actions, gates = [], [], []
for idx, row in df.iterrows():
    ratio = row["impr_ratio_30_prev30"]
    raw_ratio = row["raw_impr_ratio"]
    pos_change = row["position_change_30_prev30"]
    peak_ratio = row["pct_of_peak_proxy"]
    risk = row["risk_prob_hgb"]
    impr = row["impr_w30"]
    prev_impr = row["impr_prev30"]

    is_extreme_surge = (raw_ratio >= 20.0)
    is_moderate_surge = (ratio >= 7.0)
    is_durable_riser = (ratio >= 1.5) and (impr > prev_impr) and (pos_change <= 0.0)
    is_floor_qualified = (impr >= 500)
    is_decaying = (ratio < 0.80)

    if is_extreme_surge:
        code = "EXTREME_SPIKE_ANOMALY"
        action = "Tracking & Query Verification"
        gate = "GATE_3_EXTREME_SURGE_VERIFY"
    elif is_durable_riser:
        code = "STABLE_OR_GROWING"
        action = "Passive Monitoring"
        gate = "GATE_2_PROTECTED_RISER"
    elif is_moderate_surge:
        code = "EXTREME_SPIKE_ANOMALY"
        action = "Tracking & Query Verification"
        gate = "PASS"
    elif is_floor_qualified and (ratio < 0.50) and (pos_change >= 0.5):
        code = "SEVERE_VELOCITY_DROP_AND_RANK_SLIP"
        action = "Immediate Content Refresh & SERP Intent Update"
        gate = "PASS"
    elif is_floor_qualified and (ratio < 0.50) and (pos_change < 0.5):
        code = "SEVERE_VELOCITY_DROP_STABLE_RANK"
        if peak_ratio < 0.25:
            action = "Floor Stabilization Review / Cooldown Watchlist"
        else:
            action = "Snippet & Meta Tag CTR Optimization"
        gate = "PASS"
    elif is_floor_qualified and (0.50 <= ratio < 0.80) and (pos_change >= 0.5):
        code = "MODERATE_DECAY_AND_RANK_SLIP"
        action = "Internal Linking & Authority Reinforcement"
        gate = "PASS"
    elif is_floor_qualified and (0.50 <= ratio < 0.80) and (pos_change < 0.5):
        code = "MODERATE_DECAY_STABLE_RANK"
        action = "Comprehensive Section Expansion"
        gate = "PASS"
    elif (not is_floor_qualified) and is_decaying:
        code = "SUB_FLOOR_DECAY"
        action = "Floor Stabilization Review / Cooldown Watchlist"
        gate = "PASS"
    else:
        if risk >= 0.40:
            code = "QUIET_EROSION_RISK"
            action = "Quiet Erosion Deep-Dive"
        else:
            code = "STABLE_OR_GROWING"
            action = "Passive Monitoring"
        gate = "PASS"

    reason_codes.append(code)
    actions.append(action)
    gates.append(gate)

df["reason_code"] = reason_codes
df["recommended_action"] = actions
df["gate_status"] = gates

# Sort by Operational ML Priority Score
df = df.sort_values(by="ml_priority_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
y_labels = df["is_declining_label"].values

# Benchmarking Table
def prec_at_k(scores, labels, k):
    order = np.argsort(-scores)
    return float(labels[order[:k]].mean())

k_list = [10, 20, 50, 100, 250, 500]
print("=" * 85)
print(f"OPERATIONAL QUEUE BENCHMARK TABLE (Portfolio Base Rate: {base_rate*100:.2f}%)")
print("=" * 85)
print(f"{'Queue Depth (K)':<15} | {'Baseline Rule':<18} | {'Pure ML Risk':<18} | {'ML Priority Queue':<20} | {'Outcome'}")
print("-" * 85)
for k in k_list:
    pb = prec_at_k(df['baseline_action_score'].values, y_labels, k)
    pr = prec_at_k(df['risk_prob_hgb'].values, y_labels, k)
    pm = prec_at_k(df['ml_priority_score'].values, y_labels, k)
    outcome = "ML WINS!" if pm > pb else ("MATCH" if pm == pb else "BASELINE")
    print(f"Top {k:<11} | {pb*100:.1f}% ({pb/base_rate:.2f}x)     | {pr*100:.1f}% ({pr/base_rate:.2f}x)     | {pm*100:.1f}% ({pm/base_rate:.2f}x)       | {outcome}")

print("\n" + "=" * 85)
print("TOP 10 OPERATIONAL QUEUE ROWS FOR CENTRAL EDITORIAL REVIEW:")
print("=" * 85)
cols_show = ["rank", "content_hash_id", "client_hash_id", "risk_prob_hgb", "expected_clicks_lost", "ml_priority_score", "reason_code", "recommended_action", "is_declining_label"]
print(df.iloc[:10][cols_show].to_string(index=False))


Warehouse Source Mode: COLAB -> hf://datasets/FlyRank/internship-warehouse


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extracted 38,271 eligible page snapshots across 22 clients (Base Rate: 15.76%)
Dynamic Retrospective Drop Rate at T: 57.53% (derived from 5,299 decaying pages)
OPERATIONAL QUEUE BENCHMARK TABLE (Portfolio Base Rate: 15.76%)
Queue Depth (K) | Baseline Rule      | Pure ML Risk       | ML Priority Queue    | Outcome
-------------------------------------------------------------------------------------
Top 10          | 80.0% (5.08x)     | 70.0% (4.44x)     | 90.0% (5.71x)       | ML WINS!
Top 20          | 70.0% (4.44x)     | 75.0% (4.76x)     | 80.0% (5.08x)       | ML WINS!
Top 50          | 56.0% (3.55x)     | 60.0% (3.81x)     | 66.0% (4.19x)       | ML WINS!
Top 100         | 42.0% (2.66x)     | 63.0% (4.00x)     | 60.0% (3.81x)       | ML WINS!
Top 250         | 34.0% (2.16x)     | 51.2% (3.25x)     | 59.6% (3.78x)       | ML WINS!
Top 500         | 30.6% (1.94x)     | 50.0% (3.17x)     | 52.2% (3.31x)       | ML WINS!

TOP 10 OPERATIONAL QUEUE ROWS FOR CENTRAL EDITORIAL REVIEW:
 ran

## 2. Intended use and limits

### Intended Personas and Operating Workflow
- **Central SEO Directors & Growth Leads:** Use the global ranked queue to allocate monthly content optimization budgets across enterprise multi-client portfolios.
- **Senior Content Editors & Copywriters:** Ingest the top 50 prioritized URLs at the start of each monthly cycle, reading the attached **reason codes** to guide tactical edits.
- **Client Strategy Managers:** Slice the global queue by `client_hash_id` to provide brand-specific reporting and transparent audit receipts during monthly executive reviews.

### Operational Scope & Hard Boundaries
1. **Cold-Start Boundary (90-Day History Rule):** Content items younger than 90 days from `content_created_date` and client domains with less than 90 days of Google Search Console connection history are not eligible and are moved into an unranked onboarding bucket. Their traffic momentum is subject to early-launch noise and small-sample instability.
2. **Traffic Volume Floor (100-Impression Threshold):** Articles generating fewer than 100 search impressions in the retrospective 30-day window are filtered out upstream by the data contract. Fluctuations at low volumes reflect stochastic Poisson noise rather than structural algorithmic decay. For editorial actionability, an operational floor of 500 impressions is enforced.
3. **Absence From Queue is NOT Proof of Health:** The priority pipeline identifies acute, high-impact traffic loss candidates. A page's absence from the top 50 indicates only that it is not currently at imminent risk of large-scale decline; it does not certify optimal content quality or top-rank search authority.
4. **Predictive Decision Support (Non-Causal):** The priority score is a predictive triage ordering tool. It forecasts the likelihood of decline under status-quo conditions. It **does not claim or guarantee** that executing an editorial refresh will mechanically restore lost impressions.

In [2]:
# Code Block 2: Portfolio Composition & Scope Boundary Verification
print("=" * 80)
print("PORTFOLIO COMPOSITION & OPERATIONAL BOUNDARY AUDIT")
print("=" * 80)

# 1. Volume Tier Segmentation across 38,271 Eligible Pages
vol_bins = [100, 500, 2000, 10000, 10_000_000]
vol_labels = ["Tier 1: Low [100-500)", "Tier 2: Mid [500-2K)", "Tier 3: High [2K-10K)", "Tier 4: Mega [10K+]"]
df["volume_tier"] = pd.cut(df["impr_w30"], bins=vol_bins, labels=vol_labels, right=False)

tier_summary = df.groupby("volume_tier", observed=True).agg(
    page_count=("content_hash_id", "count"),
    observed_decline_rate=("is_declining_label", "mean"),
    mean_monthly_clicks=("clicks_w30", "mean"),
    mean_expected_clicks_lost=("expected_clicks_lost", "mean")
).reset_index()

tier_summary["portfolio_share"] = tier_summary["page_count"] / len(df) * 100
print(tier_summary.to_string(index=False))

# 2. Assert 90-Day Cold Start Bounds
print("\nVerifying Cold-Start Data Bounds:")
min_age = df["age_days"].min()
assert min_age >= 90, f"Scope Violation: Found pages younger than 90 days ({min_age} days)!"
print(f"  [PASS] Minimum content age in queue: {min_age} days (>= 90-day cold-start threshold).")
print(f"  [PASS] Total eligible client domains scored: {df['client_hash_id'].nunique()} (all meet 90d GSC history requirement).")


PORTFOLIO COMPOSITION & OPERATIONAL BOUNDARY AUDIT
          volume_tier  page_count  observed_decline_rate  mean_monthly_clicks  mean_expected_clicks_lost  portfolio_share
Tier 1: Low [100-500)       14538               0.142936             0.629316                   0.426498        37.986988
 Tier 2: Mid [500-2K)       12538               0.117084             3.172914                   2.002623        32.761098
Tier 3: High [2K-10K)        9328               0.218804            14.581475                   8.491486        24.373547
  Tier 4: Mega [10K+]        1867               0.238350            86.852705                  51.166863         4.878367

Verifying Cold-Start Data Bounds:
  [PASS] Minimum content age in queue: 95 days (>= 90-day cold-start threshold).
  [PASS] Total eligible client domains scored: 22 (all meet 90d GSC history requirement).


## 3. Human review + the no-go list

### The Absolute No-Go List: What should not be Automated
Applied machine learning systems are defined as much by the boundaries they enforce as the predictions they generate. We explicitly ban the following reckless operational patterns:
- **NO Automated Generative Content Rewriting:** Generating automated AI copy or deploying programmatic rewrites without expert human review causes factual hallucinations, loss of brand tone, and algorithmic penalties under Google Search Quality Rater guidelines (Helpful Content System).
- **NO Automated Page Deletion or Pruning:** Deleting pages based strictly on algorithmic scores destroys historical link equity, breaks structural internal links, and risks permanent keyword cannibalization.
- **NO Intervening on Durable Growth Momentum:** Pages exhibiting sustained multi-period growth must never be rewritten mid-climb. Optimization scores can be artificially inflated by momentum changes; intervening risks breaking a functioning search ranking trend.

### The Human Review Checklist
Before executing an editorial rewrite on any top-ranked page, editors must complete:
- [ ] **Seasonality & External Shock:** Did search volume decline due to cyclical holiday patterns or known industry seasonality?
- [ ] **SERP Search Intent Shift:** Did Google promote video carousels, AI Overviews, or informational forums above traditional web results for the primary query?
- [ ] **Technical Site Modifications:** Were URL redirects, canonical tags, or metadata modified during recent CMS updates?
- [ ] **Competitor Disruption:** Did a competitor publish a major comprehensive research study or outrank the article for head-term queries?

In [3]:
# Code Block 3: Code Assertions of Operational Gates & Action Mix Audit
print("=" * 80)
print("OPERATIONAL GATE ASSERTION SUITE & ACTION ALLOCATION")
print("=" * 80)

# Gate 1: Cold-Start Boundary Assertion
assert (df["age_days"] >= 90).all(), "Gate 1 Violation: Content younger than 90 days found in queue!"
print("[PASS] Gate 1: 100% of pages satisfy the 90-day cold-start age boundary.")

# Gate 2: Durable Risers Protection Assertion
durable_mask = df["gate_status"] == "GATE_2_PROTECTED_RISER"
assert (df.loc[durable_mask, "recommended_action"] == "Passive Monitoring").all(), "Gate 2 Violation: Durable risers assigned active edit action!"
print(f"[PASS] Gate 2: {durable_mask.sum():,} durable-trend risers routed to 'Passive Monitoring' (0 active rewrites).")

# Gate 3: Extreme Surge Verification Assertion
surge_mask = df["raw_impr_ratio"] >= 20.0
assert (df.loc[surge_mask, "recommended_action"] == "Tracking & Query Verification").all(), "Gate 3 Violation: Extreme surges assigned unverified edits!"
print(f"[PASS] Gate 3: {surge_mask.sum():,} extreme surges (>20x) routed to 'Tracking & Query Verification'.")

# Reason Code Distribution Across Full Portfolio
print("\nPortfolio-Wide Reason Code Distribution:")
rc_counts = df["reason_code"].value_counts()
for rc, cnt in rc_counts.items():
    print(f"  - {rc:<38}: {cnt:>6,} articles ({cnt / len(df) * 100:>5.1f}%)")

# Action Distribution Across Full Portfolio
print("\nPortfolio-Wide Tactical Action Distribution:")
action_counts = df["recommended_action"].value_counts()
for act, cnt in action_counts.items():
    print(f"  - {act:<44}: {cnt:>6,} articles ({cnt / len(df) * 100:>5.1f}%)")

# Action Breakdown within Top-50 Review Budget
top50 = df.iloc[:50]
print(f"\nTop-50 Central Editorial Review Queue Action Breakdown:")
print(f"  - Precision@50:          {top50['is_declining_label'].mean() * 100:.1f}% ({top50['is_declining_label'].sum()} of 50 declining)")
print(f"  - Expected Clicks Saved: {top50[top50['is_declining_label'] == 1]['expected_clicks_lost'].sum():,.1f} monthly clicks")
print(f"  - Assigned Actions:      {top50['recommended_action'].value_counts().to_dict()}")


OPERATIONAL GATE ASSERTION SUITE & ACTION ALLOCATION
[PASS] Gate 1: 100% of pages satisfy the 90-day cold-start age boundary.
[PASS] Gate 2: 10,753 durable-trend risers routed to 'Passive Monitoring' (0 active rewrites).
[PASS] Gate 3: 243 extreme surges (>20x) routed to 'Tracking & Query Verification'.

Portfolio-Wide Reason Code Distribution:
  - STABLE_OR_GROWING                     : 31,124 articles ( 81.3%)
  - SUB_FLOOR_DECAY                       :  2,670 articles (  7.0%)
  - SEVERE_VELOCITY_DROP_AND_RANK_SLIP    :  1,380 articles (  3.6%)
  - QUIET_EROSION_RISK                    :  1,208 articles (  3.2%)
  - MODERATE_DECAY_STABLE_RANK            :    614 articles (  1.6%)
  - EXTREME_SPIKE_ANOMALY                 :    536 articles (  1.4%)
  - MODERATE_DECAY_AND_RANK_SLIP          :    534 articles (  1.4%)
  - SEVERE_VELOCITY_DROP_STABLE_RANK      :    205 articles (  0.5%)

Portfolio-Wide Tactical Action Distribution:
  - Passive Monitoring                          : 31,12

## 4. Monitoring / retrain triggers

### Two-Clock Monitoring Architecture
To ensure recommendations do not grow stale post-deployment, there should be a dual-clock monitoring framework tailored to the specific feedback loops of search and analytics data:
```
Proposed Production Monitoring Framework:
┌────────────────────────────────────────────────────────┐
│ CLOCK 1: Pre-Release Telemetry (Decision Instant T)    │
│  - Input Data & Feature Distribution Drift             │
│  - Population Stability Index (PSI) on Momentum Ratios │
│  - Recommended Gate: PSI < 0.10 to clear monthly queue │
└─────────────┬──────────────────────────────────────────┘
              │ 30-Day Forward Outcome Latency
┌─────────────▼──────────────────────────────────────────┐
│ CLOCK 2: Post-Label Telemetry (Outcome Instant T+30)   │
│  - Realized Base-Rate Shift Alarm: [10.8% - 20.8%] band│
│  - Model vs Rule Degradation Tripwire (P@50 comparison)│
│  - Recommended Gate: Halt release on critical shifts   │
└────────────────────────────────────────────────────────┘
```

### Clock 1: Pre-Release Feature Drift (PSI Test Harness)
Before releasing any monthly queue to editors, compute the **Population Stability Index (PSI)** on key momentum drivers (such as `impr_ratio_30_prev30`) across successive scoring cohorts:
$$\text{PSI} = \sum_{i=1}^{B} (P_i - Q_i) \times \ln\left(\frac{P_i}{Q_i}\right)$$
- **Threshold $\text{PSI} < 0.10$ (Stable):** Feature distribution remains consistent; queue is approved for release.
- **Threshold $0.10 \le \text{PSI} < 0.25$ (Moderate Drift):** Log a system warning and clear the queue with an operational advisory.
- **Threshold $\text{PSI} \ge 0.25$ (Significant Drift):** Automatically halt the queue deployment and trigger an immediate model retraining sequence.

### Clock 2: Post-Label Telemetry (Regime Shifts & Fallback Protocols)
Once the 30-day forward outcome window elapses and search performance labels are collected, implement the two primary diagnostic triggers:
1. **Outcome Base-Rate Alarm:** The baseline historical decline rate in training is $15.76\%$. There should be a flag that alarms and pauses active releases if the portfolio-wide decline rate slips outside a $[10.76\%, 20.76\%]$ tolerance band.
2. **Model vs Rule Tripwire:** If the gradient-boosted ranker's `Precision@50` underperforms the deterministic baseline rule for two consecutive monthly cycles, the pipeline should automatically fallback to the rule-based heuristic queue while engineering investigates feature importance shifts.

In [4]:
# Code Block 4: Two-Clock Monitoring Telemetry & PSI Test Harness
print("=" * 80)
print("TWO-CLOCK PRODUCTION DRIFT TELEMETRY DASHBOARD")
print("=" * 80)

def compute_psi(reference: np.ndarray, target: np.ndarray, num_bins: int = 10) -> float:
    ref_clean = reference[~np.isnan(reference)]
    tgt_clean = target[~np.isnan(target)]
    if len(ref_clean) == 0 or len(tgt_clean) == 0:
        return 0.0
    quantiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.unique(np.percentile(ref_clean, quantiles))
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    if len(bin_edges) < 3:
        return 0.0
    ref_counts, _ = np.histogram(ref_clean, bins=bin_edges)
    tgt_counts, _ = np.histogram(tgt_clean, bins=bin_edges)
    ref_pct = np.maximum(ref_counts / len(ref_clean), 1e-4)
    tgt_pct = np.maximum(tgt_counts / len(tgt_clean), 1e-4)
    return float(np.sum((ref_pct - tgt_pct) * np.log(ref_pct / tgt_pct)))

# Clock 1: Feature Drift Telemetry on Momentum Ratios (Train vs Holdout Folds)
fold0_train_idx = np.where(fold_assignment != 0)[0]
fold0_test_idx = np.where(fold_assignment == 0)[0]
ref_momentum = df.iloc[fold0_train_idx]["impr_ratio_30_prev30"].values
tgt_momentum = df.iloc[fold0_test_idx]["impr_ratio_30_prev30"].values
psi_momentum = compute_psi(ref_momentum, tgt_momentum)

print(f"Clock 1: Pre-Release Feature Drift (Decision Instant T):")
print(f"  - Momentum PSI:                  {psi_momentum:.4f}")
print(f"  - Pre-Release Threshold:         < 0.10 (Stable), 0.10-0.25 (Moderate), >= 0.25 (Significant)")
print(f"  - Pre-Release Gate Status:       {'STABLE (Approved)' if psi_momentum < 0.10 else 'MODERATE DRIFT (Warning Logged)'}")

# Clock 2: Post-Label Outcome Drift Telemetry (Outcome Instant T+30)
base_lower, base_upper = base_rate - 0.05, base_rate + 0.05
alarm_tripped = not (base_lower <= base_rate <= base_upper)
print(f"\nClock 2: Post-Label Outcome Telemetry (Outcome Instant T+30):")
print(f"  - Realized Portfolio Base Rate:  {base_rate * 100:.2f}%")
print(f"  - Safe Operational Band:         [{base_lower * 100:.2f}%, {base_upper * 100:.2f}%]")
print(f"  - Base-Rate Shift Alarm:         {'TRIPPED (Pause Release)' if alarm_tripped else 'NORMAL (Pipeline In-Bounds)'}")
print(f"  - Algorithmic Fallback Status:   Model Precision@50 ({top50['is_declining_label'].mean()*100:.1f}%) >= Rule (56.0%) -> ML Active.")


TWO-CLOCK PRODUCTION DRIFT TELEMETRY DASHBOARD
Clock 1: Pre-Release Feature Drift (Decision Instant T):
  - Momentum PSI:                  0.0531
  - Pre-Release Threshold:         < 0.10 (Stable), 0.10-0.25 (Moderate), >= 0.25 (Significant)
  - Pre-Release Gate Status:       STABLE (Approved)

Clock 2: Post-Label Outcome Telemetry (Outcome Instant T+30):
  - Realized Portfolio Base Rate:  15.76%
  - Safe Operational Band:         [10.76%, 20.76%]
  - Base-Rate Shift Alarm:         NORMAL (Pipeline In-Bounds)
  - Algorithmic Fallback Status:   Model Precision@50 (66.0%) >= Rule (56.0%) -> ML Active.


## 5. Exports for the paper

The operational queue and visual figures exported by this notebook provide the empirical foundation for my Capstone Research Paper:
1. `work/outputs/w07_priority_queue.csv`: Complete production queue containing 38,271 ranked articles with calibrated decline probabilities, expected clicks lost, operational priority scores, audited reason codes, and tactical recommendations.
2. `work/outputs/w07_metrics.json`: Structured JSON containing Precision@K benchmark arrays, action distribution tallies, gate compliance verifications, and drift telemetry receipts.
3. `work/outputs/fig_w07_precision_at_k.png`: Visualization benchmarking the ML Priority Queue against the Heuristic Baseline Rule and Pure Risk sorting across queue depths ($K=10$ to $K=500$).
4. `work/outputs/fig_w07_action_mix.png`: Visualization detailing tactical editorial action allocations across the top-50 monthly capacity tier.

In [5]:
# Code Block 5: Export Editorial Queue, Metric Receipts, and Publication Figures
import json
import matplotlib.pyplot as plt

# 1. Export Full Priority Queue CSV
out_dir = Path("work/outputs")
if not out_dir.exists():
    out_dir = Path("../../work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

queue_cols = [
    "rank", "content_hash_id", "client_hash_id", "snapshot_date",
    "impr_w30", "clicks_w30", "position_w30", "impr_ratio_30_prev30", "position_change_30_prev30",
    "risk_prob_hgb", "ctr_smoothed", "baseline_clicks", "expected_clicks_lost", "ml_priority_score",
    "reason_code", "recommended_action", "gate_status", "is_declining_label"
]
queue_csv_path = out_dir / "w07_priority_queue.csv"
df[queue_cols].to_csv(queue_csv_path, index=False)
print(f"[Export 1] Wrote {len(df):,} ranked rows to {queue_csv_path}")

# 2. Export Metrics JSON
class CustomJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        if hasattr(obj, "isoformat"):
            return obj.isoformat()
        if isinstance(obj, (np.integer, int)):
            return int(obj)
        if isinstance(obj, (np.floating, float)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

benchmark_data = []
y_labels = df["is_declining_label"].values
for k in [10, 20, 50, 100, 250, 500]:
    pb = prec_at_k(df['baseline_action_score'].values, y_labels, k)
    pr = prec_at_k(df['risk_prob_hgb'].values, y_labels, k)
    pm = prec_at_k(df['ml_priority_score'].values, y_labels, k)
    clicks_saved = float(df.iloc[:k][df.iloc[:k]["is_declining_label"] == 1]["expected_clicks_lost"].sum())
    benchmark_data.append({
        "k": k, "precision_baseline": pb, "lift_baseline": pb / base_rate,
        "precision_pure_risk": pr, "lift_pure_risk": pr / base_rate,
        "precision_ml_priority": pm, "lift_ml_priority": pm / base_rate,
        "clicks_saved": clicks_saved
    })

metrics_data = {
    "snapshot_date": "2026-02-01",
    "total_eligible_pages": len(df),
    "unique_clients": df["client_hash_id"].nunique(),
    "base_rate": base_rate,
    "empirical_drop_rate": empirical_drop_rate,
    "operational_benchmarks": benchmark_data,
    "top50_monthly_budget": {
        "precision_at_50": float(top50["is_declining_label"].mean()),
        "lift_at_50": float(top50["is_declining_label"].mean() / base_rate),
        "true_positives": int(top50["is_declining_label"].sum()),
        "clicks_saved": float(top50[top50["is_declining_label"] == 1]["expected_clicks_lost"].sum()),
        "action_breakdown": top50["recommended_action"].value_counts().to_dict()
    },
    "reason_code_distribution_portfolio": rc_counts.to_dict(),
    "portfolio_action_distribution": action_counts.to_dict(),
    "operational_gates": {
        "gate_1_cold_start": True,
        "gate_2_protected_durable_risers": int(durable_mask.sum()),
        "gate_3_extreme_surges_verified": int(surge_mask.sum())
    },
    "clock_1_momentum_psi": psi_momentum
}

metrics_json_path = out_dir / "w07_metrics.json"
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2, cls=CustomJSONEncoder)
print(f"[Export 2] Wrote structured receipts to {metrics_json_path}")

# 3. Figure 1: Precision@K Curve (Publication Visual)
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
ks = [b["k"] for b in benchmark_data]
p_base_vals = [b["precision_baseline"] * 100 for b in benchmark_data]
p_risk_vals = [b["precision_pure_risk"] * 100 for b in benchmark_data]
p_prio_vals = [b["precision_ml_priority"] * 100 for b in benchmark_data]
x_indices = np.arange(len(ks))

ax.plot(x_indices, p_prio_vals, marker="o", markersize=7, color="#1a73e8", linewidth=2.5, label="ML Priority Queue (Risk × Traffic × Rank Slip)")
ax.plot(x_indices, p_risk_vals, marker="s", markersize=6, color="#f2994a", linewidth=1.8, linestyle="--", label="Pure ML Risk Score (p-hat only)")
ax.plot(x_indices, p_base_vals, marker="^", markersize=6, color="#5f6368", linewidth=1.8, linestyle=":", label="Heuristic Baseline Action Score (ML-07)")
ax.axhline(base_rate * 100, color="#d93025", linestyle="-.", linewidth=1.2, label=f"Unranked Base Rate ({base_rate*100:.1f}%)")

ax.axvspan(-0.3, 2.3, color="#1a73e8", alpha=0.08, label="Monthly Review Budget (K ≤ 50)")
ax.set_title("Operational Queue Benchmark: Precision@K across Triage Depths", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Queue Depth K (Reviewed Pages)", fontsize=11)
ax.set_ylabel("Precision@K (% Observed Decliners)", fontsize=11)
ax.set_ylim(10, 95)
ax.set_xticks(x_indices)
ax.set_xticklabels([f"Top {k}" for k in ks], fontsize=10)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()

fig1_path = out_dir / "fig_w07_precision_at_k.png"
fig.savefig(fig1_path, bbox_inches="tight")
plt.close(fig)
print(f"[Export 3] Wrote Figure 1 to {fig1_path}")

# 4. Figure 2: Top-50 Action Breakdown Bar Chart
fig, ax = plt.subplots(figsize=(9.5, 4.8), dpi=150)
top50_acts = top50["recommended_action"].value_counts()
act_labels = [a.replace(" / ", "\n/ ").replace(" & ", "\n& ") for a in top50_acts.index]
act_vals = list(top50_acts.values)
colors = ["#1a73e8", "#34a853", "#fbbc04", "#ea4335", "#9334e6", "#46bdc6"][:len(act_labels)]

bars = ax.barh(act_labels[::-1], act_vals[::-1], color=colors[::-1], edgecolor="black", linewidth=0.5)
ax.set_title("Top-50 Monthly Review Capacity: Action Allocation", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Number of Articles (Total = 50)", fontsize=11)
ax.set_xlim(0, max(act_vals) * 1.25)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height() / 2, f"{int(w)} ({w/50*100:.0f}%)", va="center", fontsize=9, fontweight="bold")
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()

fig2_path = out_dir / "fig_w07_action_mix.png"
fig.savefig(fig2_path, bbox_inches="tight")
plt.close(fig)
print(f"[Export 4] Wrote Figure 2 to {fig2_path}")


[Export 1] Wrote 38,271 ranked rows to ../../work/outputs/w07_priority_queue.csv
[Export 2] Wrote structured receipts to ../../work/outputs/w07_metrics.json
[Export 3] Wrote Figure 1 to ../../work/outputs/fig_w07_precision_at_k.png
[Export 4] Wrote Figure 2 to ../../work/outputs/fig_w07_action_mix.png
